# Learning path-dependent payoffs
In this notebook we run a toy-example to test our functional Nadaraya-Watson estimator for the learning of $$X^H|_{[0,T]} \mapsto \Psi(X^H|_{[0,T]}),$$ where $X^H$ is a fractional Brownian motion with Hurst parameter $H$, and $\Psi$ represents a path-dependent option payoff. The payoffs that are considered here are $$\textbf{Asian-Options:} \quad \Psi(x|_{[0,T]}) = \left ( \frac{1}{T} \int_0^Tx_s ds-K \right )_+,$$ $$\textbf{Lookback-Options:} \quad \Psi(x|_{[0,T]}) = \left (\max_{0 \leq t \leq T}x_t-K \right)_+,$$ $$\textbf{Barrier-Options:} \quad \Psi(x|_{[0,T]})= 1_{\{\max_{0\leq t \leq T} X_t < B\}}(x_T-K)_+$$

We compare the suggested signature-metric estimator with various other norms on the path-space.

In [ ]:
import sys
import os
# Add parent directory to path so we can import from nwsig package
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import iisignature as ii
import numpy as np
import matplotlib.pyplot as plt

# Import from nwsig package
from nwsig import NWSignatureRegressor, fBm_sampling
from nwsig.metrics import MSE, trunc_sig_metric, supremum_metric, full_sig_metric

In [ ]:
#Ridge Regression wrappers

from sklearn.linear_model import RidgeCV

class SimpleRidgeOnSignature:
    def __init__(self, signature_level, alphas=None, cv=5):
        self.signature_level = signature_level
        self.alphas = np.logspace(-8, 3, 25) if alphas is None else alphas
        self.cv = cv

    def fit(self, paths, y, x0=None):
        sig = ii.sig(paths, self.signature_level)  # (M, sigdim) if your ii.sig supports 3D input
        self.regr_ = RidgeCV(alphas=self.alphas, fit_intercept=True, cv=self.cv).fit(sig, y)
        self.best_alpha_ = self.regr_.alpha_
        return self

    def predict(self, paths, x0=None):
        sig = ii.sig(paths, self.signature_level)
        return self.regr_.predict(sig)


In [ ]:
#Main function running the experiments for range of methods
import time as time

def run_fBm_experiments(M_values, N=400, T=1, x0=1,
                        truncation_levels=[1,2,3,4,5,6],
                        p_values=[1,2,4,6,8],
                        H=[0.2,0.35,0.5,0.75,0.9],K=0.5,B = 1.4):
    """
    Run path-dependent options learning experiments
    """
    # Get the maximum M to generate data only once
    max_M = max(M_values)*2

    # Define all methods (robust signatures with different truncation levels + standard metrics)
    method_configs = []

    # Add robust signature methods with different truncation levels
    for k in truncation_levels:
        method_configs.append(('robust_signature', k))
        method_configs.append(('classical_signature', k))
        method_configs.append(('ridge_signature', k))

    for p in p_values:
        method_configs.append(('p_variation', p))

    # Add standard metrics (no truncation level dependency)
    method_configs.append(('supremum', None))
    method_configs.append(('l1', None))
    method_configs.append(('l2', None))

    # Create method names for results
    method_names = []
    for method, k in method_configs:
        if k is not None:
            method_names.append(f"{method}_K{k}")
        else:
            method_names.append(method)

    payoff_names = ["asian", "lookback", "barrier"]

    # Initialize results storage
    num_methods = len(method_configs)
    mse_results = np.zeros((len(H), len(payoff_names), len(M_values), num_methods))
    time_results = np.zeros((len(H), len(payoff_names), len(M_values), num_methods))
    best_bandwidths = {}

    # Generate time grid
    tt = np.linspace(0, T, N+1)

    fBm_augmented = []  # paths for each Hurst parameter
    Y_a = []            # Asian payoffs for each Hurst parameter
    Y_l = []            # Lookback payoffs for each Hurst parameter
    Y_b = []            # Barrier payoffs for each Hurst parameter

    # Simulate fBm + build payoffs once for each H
    for h_idx in range(len(H)):
        fbm = fBm_sampling(max_M, N, H[h_idx])[0]  # expected shape (max_M, N+1)

        fbm_aug = np.zeros((max_M, N+1, 2))
        fbm_aug[:, :, 0] = tt
        fbm_aug[:, :, 1] = fbm  # interpret fbm as log-price

        fBm_augmented.append(fbm_aug)

        # NOTE: expects K and B are defined in the surrounding scope
        Y_a.append(np.maximum(np.mean(fBm_augmented[h_idx][:, :, 1], axis=1) - K, 0.0))
        Y_l.append(np.maximum(np.max(fBm_augmented[h_idx][:, :, 1], axis=1) - K, 0.0))
        Y_b.append((np.max(fBm_augmented[h_idx][:, :, 1], axis=1) < B).astype(np.float64)
                   * np.maximum(fBm_augmented[h_idx][:, -1, 1] - K, 0.0))

    # Run experiments for each H and each payoff
    for h_idx in range(len(H)):
        print(f"\n==============================")
        print(f"Running H = {H[h_idx]}")
        print(f"==============================")

        Y_targets = [Y_a[h_idx], Y_l[h_idx], Y_b[h_idx]]

        for payoff_idx, payoff_name in enumerate(payoff_names):
            print(f"\n--- Payoff: {payoff_name} ---")
            Y = Y_targets[payoff_idx]

            # Run experiments for each M value
            for m_idx, M in enumerate(M_values):
                print(f"\nRunning experiment with M = {M}")

                # Use subset of data
                W_aug_subset = fBm_augmented[h_idx][:M]
                Y_subset = Y[:M]

                # Calculate training/testing split
                M_training = M

                # Run for each method configuration
                for method_idx, (method, k) in enumerate(method_configs):
                    start_time = time.time()

                    if method == 'classical_signature':
                        print(f"  Running classical signature method with K={k}")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[0.1, 0.05, 0.01, 0.005, 0.001],
                            signature_level=k,
                            robust=False,
                            robust_a_grid=[0.5, 1],
                            robust_C_grid=[2,10,20],
                            initial_condition=False,
                            metric_choice="signature_metric"
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    if method == 'robust_signature':
                        print(f"  Running robust signature method with K={k}")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[0.1, 0.05, 0.01, 0.005, 0.001],
                            signature_level=k,
                            robust=True,
                            robust_a_grid=[0.5, 1],
                            robust_C_grid=[2,10,20],
                            initial_condition=False,
                            metric_choice="signature_metric"
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    if method == 'p_variation':
                        print(f"  Running p variation method with K={k}")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[2,1, 0.1, 0.01, 0.001,0.0001,0.00001],
                            signature_level=max(truncation_levels),  # Use the highest level for fairness
                            robust=False,
                            initial_condition=False,
                            metric_choice="pvar",
                            p_param=k
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    if method == 'ridge_signature':
                        print(f"  Running parametric Ridge regression at level K={k}")
                        NW = SimpleRidgeOnSignature(signature_level=k, alphas=np.logspace(-8, 3, 25), cv=5)\
                            .fit(W_aug_subset[:M_training,:,:], Y_subset[:M_training])

                    elif method == 'supremum':
                        print("  Running supremum metric")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[10, 1, 0.1, 0.01, 0.001],
                            signature_level=max(truncation_levels),  # Use the highest level for fairness
                            robust=False,
                            initial_condition=False,
                            metric_choice="supremum_metric"
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    elif method == 'l1':
                        print("  Running L1 metric")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[10, 1, 0.1, 0.01, 0.001],
                            metric_choice="l1"
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    elif method == 'l2':
                        print("  Running L2 metric")
                        NW = NWSignatureRegressor(
                            input_type='paths',
                            bandwidth_grid=[10, 1, 0.1, 0.01, 0.001],
                            metric_choice="l2"
                        ).fit(W_aug_subset[0:M_training,:,:],
                              Y_subset[0:M_training])

                    # Make predictions and calculate MSE (same testing convention as before)
                    test_start = M_values[-1]
                    W_test = fBm_augmented[h_idx][test_start:,:,:]
                    y_test = Y[test_start:]

                    predictions = NW.predict(W_test)
                    mse_results[h_idx, payoff_idx, m_idx, method_idx] = MSE(predictions, y_test)
                    time_results[h_idx, payoff_idx, m_idx, method_idx] = time.time() - start_time

                    # Save bandwidth/alpha if available
                    if hasattr(NW, 'best_bandwidth_'):
                        best_bandwidths[f'H{H[h_idx]}_{payoff_name}_{method_names[method_idx]}_M{M}'] = NW.best_bandwidth_
                    if hasattr(NW, "best_alpha_"):
                        best_bandwidths[f'H{H[h_idx]}_{payoff_name}_ridge_signature_K{k}_M{M}'] = NW.best_alpha_

                    print(f"    MSE: {mse_results[h_idx, payoff_idx, m_idx, method_idx]:.6f}, "
                          f"Time: {time_results[h_idx, payoff_idx, m_idx, method_idx]:.2f}s")

    return {
        'M_values': M_values,
        'H_values': H,
        'payoff_names': payoff_names,
        'method_names': method_names,
        'mse_results': mse_results,
        'time_results': time_results,
        'best_bandwidths': best_bandwidths
    }



In [ ]:
# Define the M values to test
M_values = [2**i for i in range(4,14)]  # You can adjust these values
truncation_levels = [2,3,4,5]
p_values = [] #We exclude here p-variation as it takes to much time

# Run the experiments
results = run_fBm_experiments(M_values,H=[0.3], truncation_levels=truncation_levels,p_values = p_values)


In [ ]:
#define methods
method_configs = []
    
    # Add robust signature methods with different truncation levels
for k in truncation_levels:
        method_configs.append(('robust_signature', k))
        method_configs.append(('classical_signature', k))
for p in p_values:
        method_configs.append(('p_variation', p))
        
    
# Add standard metrics (no truncation level dependency)
method_configs.append(('supremum', None))
method_configs.append(('l1', None))
method_configs.append(('l2', None))
    
    # Create method names for results
method_names = []
for method, k in method_configs:
        if k is not None:
            method_names.append(f"{method}_K{k}")
        else:
            method_names.append(method)


In [ ]:
#function for building performance plots
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_payoff_grouped_legend(results, payoff_name, truncation_levels, p_values):
    M_values     = results['M_values']
    H_values     = results['H_values']
    payoff_names = results['payoff_names']
    method_names = results['method_names']
    mse_results  = results['mse_results']  # (H, payoff, M, method)

    payoff_idx = payoff_names.index(payoff_name)

    def midx(name):
        return method_names.index(name) if name in method_names else None

    sup_idx = midx('supremum')
    l1_idx  = midx('l1')
    l2_idx  = midx('l2')

    nH = len(H_values)
    if nH == 1:
        fig, axes = plt.subplots(1, 1, figsize=(14, 6))
        axes = [axes]
    else:
        fig, axes = plt.subplots(1, nH, figsize=(6.5*nH, 6), sharey=True)

    # --- grouped color palettes by method ---
    K = list(truncation_levels)
    colors_robust   = plt.cm.Blues   (np.linspace(0.35, 0.95, len(K)))
    colors_classic  = plt.cm.Oranges (np.linspace(0.35, 0.95, len(K)))
    colors_ridge    = plt.cm.Greens  (np.linspace(0.35, 0.95, len(K)))

    p_markers = ['d', '*', 'v', 'o', 's', '^', 'x']
    p_colors = plt.cm.Purples(np.linspace(0.35, 0.95, len(p_values)))

    # --- plot curves ---
    for h_idx, H in enumerate(H_values):
        ax = axes[h_idx]
        mse_hp = mse_results[h_idx, payoff_idx, :, :]  # (len(M), num_methods)

        for i, k in enumerate(K):
            name = f'robust_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, mse_hp[:, idx], marker='o', color=colors_robust[i])

            name = f'classical_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, mse_hp[:, idx], marker='o', color=colors_classic[i])

            name = f'ridge_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, mse_hp[:, idx], marker='o', color=colors_ridge[i])

        for i, p in enumerate(p_values):
            name = f'p_variation_K{p}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, mse_hp[:, idx],
                          marker=p_markers[i % len(p_markers)],
                          linestyle='-.', color=p_colors[i])

        if sup_idx is not None:
            ax.loglog(M_values, mse_hp[:, sup_idx], marker='s', linestyle='--', color='red')
        if l1_idx is not None:
            ax.loglog(M_values, mse_hp[:, l1_idx], marker='^', linestyle='--', color='black')
        if l2_idx is not None:
            ax.loglog(M_values, mse_hp[:, l2_idx], marker='x', linestyle='--', color='gray')

        ax.set_title(f'H = {H:.2f}', fontsize=14)
        ax.grid(True, which="both", ls="--", alpha=0.6)
        ax.set_xlabel('M', fontsize=12)

    axes[0].set_ylabel('Mean Squared Error', fontsize=12)
    fig.suptitle(f'Payoff: {payoff_name}', fontsize=16)

    # --- grouped legend: headers + K entries ---
    # headers are "dummy" handles so the legend reads nicely
    header = lambda txt: Line2D([], [], linestyle='None', marker='', label=txt)

    handles = []
    handles += [header("Robust signature")]
    handles += [Line2D([], [], color=colors_robust[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    handles += [header("Classical signature")]
    handles += [Line2D([], [], color=colors_classic[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    handles += [header("Ridge on signature")]
    handles += [Line2D([], [], color=colors_ridge[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    if len(p_values) > 0:
        handles += [header("p-variation")]
        handles += [Line2D([], [], color=p_colors[i], marker=p_markers[i % len(p_markers)],
                           linestyle='-.', label=f"p={p}") for i, p in enumerate(p_values)]

    handles += [header("Standard metrics")]
    if sup_idx is not None:
        handles += [Line2D([], [], color='red', marker='s', linestyle='--', label='supremum')]
    if l1_idx is not None:
        handles += [Line2D([], [], color='black', marker='^', linestyle='--', label=r'$L^1$')]
    if l2_idx is not None:
        handles += [Line2D([], [], color='gray', marker='x', linestyle='--', label=r'$L^2$')]

    # place legend outside
    fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
               fontsize=10, frameon=True, handlelength=2.0)

    fig.tight_layout(rect=(0, 0, 0.82, 0.92))
    plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def plot_runtime_grouped_legend(results, payoff_name, truncation_levels, p_values, savepath=None):
    M_values     = results['M_values']
    H_values     = results['H_values']
    payoff_names = results['payoff_names']
    method_names = results['method_names']
    time_results = results['time_results']  # (H, payoff, M, method)

    payoff_idx = payoff_names.index(payoff_name)

    def midx(name):
        return method_names.index(name) if name in method_names else None

    sup_idx = midx('supremum')
    l1_idx  = midx('l1')
    l2_idx  = midx('l2')

    nH = len(H_values)
    if nH == 1:
        fig, axes = plt.subplots(1, 1, figsize=(14, 6))
        axes = [axes]
    else:
        fig, axes = plt.subplots(1, nH, figsize=(6.5*nH, 6), sharey=True)

    K = list(truncation_levels)
    colors_robust  = plt.cm.Blues   (np.linspace(0.35, 0.95, len(K)))
    colors_classic = plt.cm.Oranges (np.linspace(0.35, 0.95, len(K)))
    colors_ridge   = plt.cm.Greens  (np.linspace(0.35, 0.95, len(K)))

    p_markers = ['d', '*', 'v', 'o', 's', '^', 'x']
    p_colors = plt.cm.Purples(np.linspace(0.35, 0.95, len(p_values)))

    for h_idx, H in enumerate(H_values):
        ax = axes[h_idx]
        t_hp = time_results[h_idx, payoff_idx, :, :]  # (len(M), num_methods)

        for i, k in enumerate(K):
            name = f'robust_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, t_hp[:, idx], marker='o', color=colors_robust[i])

            name = f'classical_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, t_hp[:, idx], marker='o', color=colors_classic[i])

            name = f'ridge_signature_K{k}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, t_hp[:, idx], marker='o', color=colors_ridge[i])

        for i, p in enumerate(p_values):
            name = f'p_variation_K{p}'
            if name in method_names:
                idx = method_names.index(name)
                ax.loglog(M_values, t_hp[:, idx],
                          marker=p_markers[i % len(p_markers)],
                          linestyle='-.', color=p_colors[i])

        if sup_idx is not None:
            ax.loglog(M_values, t_hp[:, sup_idx], marker='s', linestyle='--', color='red')
        if l1_idx is not None:
            ax.loglog(M_values, t_hp[:, l1_idx], marker='^', linestyle='--', color='black')
        if l2_idx is not None:
            ax.loglog(M_values, t_hp[:, l2_idx], marker='x', linestyle='--', color='gray')

        ax.set_title(f'H = {H:.2f}', fontsize=14)
        ax.grid(True, which="both", ls="--", alpha=0.6)
        ax.set_xlabel('M', fontsize=12)

    axes[0].set_ylabel('Runtime (seconds)', fontsize=12)
    fig.suptitle(f'Runtime — Payoff: {payoff_name}', fontsize=16)

    header = lambda txt: Line2D([], [], linestyle='None', marker='', label=txt)

    handles = []
    handles += [header("Robust signature")]
    handles += [Line2D([], [], color=colors_robust[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    handles += [header("Classical signature")]
    handles += [Line2D([], [], color=colors_classic[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    handles += [header("Ridge on signature")]
    handles += [Line2D([], [], color=colors_ridge[i], marker='o', linestyle='-',
                       label=f"K={k}") for i, k in enumerate(K)]

    if len(p_values) > 0:
        handles += [header("p-variation")]
        handles += [Line2D([], [], color=p_colors[i], marker=p_markers[i % len(p_markers)],
                           linestyle='-.', label=f"p={p}") for i, p in enumerate(p_values)]

    handles += [header("Standard metrics")]
    if sup_idx is not None:
        handles += [Line2D([], [], color='red', marker='s', linestyle='--', label='supremum')]
    if l1_idx is not None:
        handles += [Line2D([], [], color='black', marker='^', linestyle='--', label=r'$L^1$')]
    if l2_idx is not None:
        handles += [Line2D([], [], color='gray', marker='x', linestyle='--', label=r'$L^2$')]

    fig.legend(handles=handles, loc='center left', bbox_to_anchor=(1.02, 0.5),
               fontsize=10, frameon=True, handlelength=2.0)

    fig.tight_layout(rect=(0, 0, 0.82, 0.92))

    if savepath is not None:
        plt.savefig(savepath, bbox_inches="tight")

    plt.show()


# RMSE comparison for each option, H=0.3


In [ ]:
plot_payoff_grouped_legend(results, "asian", truncation_levels, p_values)

In [ ]:
plot_payoff_grouped_legend(results, "lookback", truncation_levels, p_values)

In [ ]:
plot_payoff_grouped_legend(results, "barrier", truncation_levels, p_values)


# Runtime Comparison

In [ ]:
plot_runtime_grouped_legend(results, "asian", truncation_levels, p_values)


In [ ]:
plot_runtime_grouped_legend(results, "lookback", truncation_levels, p_values)

In [ ]:
plot_runtime_grouped_legend(results, "barrier", truncation_levels, p_values)

In [ ]:
#Create table for the paper:
import numpy as np
import pandas as pd

def make_option_tables(results, truncation_levels=(2,3,4), p_values=(1,2,4),
                       H_select=None, include_ridge=True):
    """
    Builds one table per payoff (and per H if multiple Hs are present).
    Values are RMSE = sqrt(MSE). Adds a final row with max runtime over M for each column.
    """
    M_values     = results['M_values']
    H_values     = results['H_values']
    payoff_names = results['payoff_names']
    method_names = results['method_names']
    mse          = results['mse_results']   # (H, payoff, M, method)
    runt         = results['time_results']  # (H, payoff, M, method)

    def midx(name):
        return method_names.index(name) if name in method_names else None

    sup_idx = midx('supremum')
    l1_idx  = midx('l1')
    l2_idx  = midx('l2')

    H_indices = list(range(len(H_values))) if H_select is None else [H_values.index(h) for h in H_select]

    tables = {}  # key: (payoff, H) -> DataFrame

    for payoff_idx, payoff in enumerate(payoff_names):
        for h_idx in H_indices:
            cols = {}

            # signatures (robust/classical) + optionally ridge
            for K in truncation_levels:
                rname = f'robust_signature_K{K}'
                cname = f'classical_signature_K{K}'
                ridname = f'ridge_signature_K{K}'

                if rname in method_names:
                    cols[rf"$\mathrm{{RSig}}^{{\leq {K}}}$"] = np.sqrt(mse[h_idx, payoff_idx, :, method_names.index(rname)])
                if cname in method_names:
                    cols[rf"$\mathrm{{Sig}}^{{\leq {K}}}$"]  = np.sqrt(mse[h_idx, payoff_idx, :, method_names.index(cname)])
                if include_ridge and (ridname in method_names):
                    cols[rf"$\mathrm{{RidgeSig}}^{{\leq {K}}}$"] = np.sqrt(mse[h_idx, payoff_idx, :, method_names.index(ridname)])

            # p-variation
            for p in p_values:
                pname = f'p_variation_K{p}'
                if pname in method_names:
                    cols[rf"${p}$-var"] = np.sqrt(mse[h_idx, payoff_idx, :, method_names.index(pname)])

            # standard metrics
            if l1_idx is not None:
                cols[r"$L^1$"] = np.sqrt(mse[h_idx, payoff_idx, :, l1_idx])
            if l2_idx is not None:
                cols[r"$L^2$"] = np.sqrt(mse[h_idx, payoff_idx, :, l2_idx])
            if sup_idx is not None:
                cols[r"$\mathrm{sup}$"] = np.sqrt(mse[h_idx, payoff_idx, :, sup_idx])

            # build df
            df = pd.DataFrame(cols, index=M_values)
            df.index.name = r"$M$"

            # max runtime row (max over M)
            max_times = []
            for c in df.columns:
                # map the pretty column header back to method index
                if c.startswith(r"$\mathrm{RSig}"):
                    K = int(c.split(r"\leq ")[1].split("}")[0])
                    mi = method_names.index(f'robust_signature_K{K}')
                elif c.startswith(r"$\mathrm{Sig}"):
                    K = int(c.split(r"\leq ")[1].split("}")[0])
                    mi = method_names.index(f'classical_signature_K{K}')
                elif c.startswith(r"$\mathrm{RidgeSig}"):
                    K = int(c.split(r"\leq ")[1].split("}")[0])
                    mi = method_names.index(f'ridge_signature_K{K}')
                elif c.endswith(r"-var"):
                    p = int(c.split("$")[1])
                    mi = method_names.index(f'p_variation_K{p}')
                elif c == r"$L^1$":
                    mi = l1_idx
                elif c == r"$L^2$":
                    mi = l2_idx
                elif c == r"$\mathrm{sup}$":
                    mi = sup_idx
                else:
                    mi = None

                if mi is None:
                    max_times.append(np.nan)
                else:
                    max_times.append(np.max(runt[h_idx, payoff_idx, :, mi]))

            df.loc["max. run time (s)"] = max_times
            tables[(payoff, H_values[h_idx])] = df

    return tables


## Generate tables as in the paper

In [ ]:
#ASIAN Options
tables = make_option_tables(results, truncation_levels=(2,3,4,5), p_values=(1,2,4,6,8), include_ridge=True)
# pick which H you want:
display(tables[("asian", results["H_values"][0])])


In [ ]:
#LOOKBACK options
display(tables[("lookback", results["H_values"][0])])


In [ ]:
display(tables[("barrier", results["H_values"][0])])


## Converte tables to LateX:

In [ ]:
def df_to_latex_with_midrule(df, filename="convergence_table.tex", decimals=3):
    latex_table = df.round(decimals).to_latex(escape=False, float_format=f"%.{decimals}f")
    lines = latex_table.splitlines()

    for i, line in enumerate(lines):
        if "max. run time (s)" in line:
            lines.insert(i, r"\midrule")
            break

    latex_table = "\n".join(lines)
    print(latex_table)
    with open(filename, "w") as f:
        f.write(latex_table)
    return latex_table

# example:
# latex = df_to_latex_with_midrule(df, filename="asian_H03_table.tex", decimals=3)


In [ ]:
for (payoff, H), df in tables.items():
    fname = f"table_{payoff}_H{H:.2f}.tex".replace(".", "p")
    df_to_latex_with_midrule(df, filename=fname, decimals=3)
